In [14]:
# CÉLULA 1: Instalação das dependências
!pip install highspy pandas numpy -q

In [15]:
# CÉLULA 2: EXPERIMENTO 2 (Avançado) - LB-EDD+ (Local Branching + Fix-and-Optimize)
import os
import time
import numpy as np
import highspy

def build_ulsr_model(datafile):
    """Lê o arquivo de dados e constrói o modelo MIP base no HiGHS."""
    with open(datafile, 'r') as file:
        linhas = [a.strip() for a in file.readlines() if a.strip()]

    N = int(linhas[0])
    FR_val, F_val  = float(linhas[1]), float(linhas[2])
    HR_val, H_val  = float(linhas[3]), float(linhas[4])

    D = np.array([float(v) for v in linhas[5].split()], dtype=float)
    R = np.array([float(v) for v in linhas[6].split()], dtype=float)

    P, PR = np.zeros(N), np.zeros(N)
    F, FR = np.full(N, F_val), np.full(N, FR_val)
    H, HR = np.full(N, H_val), np.full(N, HR_val)

    D_suffix = np.cumsum(D[::-1])[::-1]
    R_prefix = np.cumsum(R)

    h = highspy.Highs()
    inf = highspy.kHighsInf

    idx_x  = list(range(0, N))
    idx_s  = list(range(N, 2*N))
    idx_y  = list(range(2*N, 3*N))
    idx_xr = list(range(3*N, 4*N))
    idx_sr = list(range(4*N, 5*N))
    idx_yr = list(range(5*N, 6*N))

    for i in range(N): h.addCol(P[i], 0.0, inf, 0, [], [])
    for i in range(N): h.addCol(H[i], 0.0, inf, 0, [], [])
    for i in range(N): h.addCol(F[i], 0.0, 1.0, 0, [], [])
    for i in range(N): h.addCol(PR[i], 0.0, inf, 0, [], [])
    for i in range(N): h.addCol(HR[i], 0.0, inf, 0, [], [])
    for i in range(N): h.addCol(FR[i], 0.0, 1.0, 0, [], [])

    for i in range(N):
        h.changeColIntegrality(idx_y[i], highspy.HighsVarType.kInteger)
        h.changeColIntegrality(idx_yr[i], highspy.HighsVarType.kInteger)

    h.addRow(D[0], D[0], 3, [idx_x[0], idx_xr[0], idx_s[0]], [1.0, 1.0, -1.0])
    for i in range(1, N):
        h.addRow(D[i], D[i], 4, [idx_s[i-1], idx_x[i], idx_xr[i], idx_s[i]], [1.0, 1.0, 1.0, -1.0])

    h.addRow(-R[0], -R[0], 2, [idx_xr[0], idx_sr[0]], [-1.0, -1.0])
    for i in range(1, N):
        h.addRow(-R[i], -R[i], 3, [idx_sr[i-1], idx_xr[i], idx_sr[i]], [1.0, -1.0, -1.0])

    for i in range(N):
        big_m = float(D_suffix[i])
        h.addRow(-inf, 0.0, 2, [idx_x[i], idx_y[i]], [1.0, -big_m])

    for i in range(N):
        cap_max = min(float(D_suffix[i]), float(R_prefix[i]))
        h.addRow(-inf, 0.0, 2, [idx_xr[i], idx_yr[i]], [1.0, -cap_max])

    h.addRow(0.0, 0.0, 1, [idx_s[N-1]], [1.0])

    binary_indices = idx_y + idx_yr
    return h, binary_indices, N

def ulsr_advanced_local_branching(datafile, time_limit=300.0, init_time=15.0, sub_time_limit=4.0,
                                   k_min=6, k_max=14, window_size=12, debug=False):
    """
    EXPERIMENTO 2 (LB-EDD+): Local Branching com Expansão Dinâmica e Polimento Fix-and-Optimize.
    """
    start_time = time.time()

    # 1. Aquecimento com Tunagem do Solver (15s)
    h_init, binary_indices, N = build_ulsr_model(datafile)
    h_init.setOptionValue("output_flag", debug)
    h_init.setOptionValue("presolve", "on")
    h_init.setOptionValue("mip_heuristics_effort", 0.2)
    h_init.setOptionValue("time_limit", float(min(init_time, time_limit)))
    h_init.run()

    sol_init = h_init.getSolution()
    info_init = h_init.getInfo()

    if not sol_init.value_valid:
        return {
            'INSTANCIA': os.path.basename(datafile),
            'BOUND': None,
            'OBJVAL': None,
            'GAP': None,
            'TIME': time.time() - start_time,
            'OTIMALIDADE': 'Infeaso'
        }

    best_obj = info_init.objective_function_value
    best_bound = getattr(info_init, "mip_dual_bound", None)
    incumbent_sol = np.round([sol_init.col_value[idx] for idx in binary_indices])

    k = k_min

    # 2. Fase 1: Local Branching Dinâmico (Busca Global de Setups)
    while (time.time() - start_time) < (time_limit * 0.7):  # Dedica até 70% do tempo total ao LB
        remaining_time = time_limit - (time.time() - start_time)
        if remaining_time <= 1.0:
            break

        h, bin_idxs, _ = build_ulsr_model(datafile)
        h.setOptionValue("output_flag", debug)
        h.setOptionValue("presolve", "on")
        h.setOptionValue("time_limit", float(min(sub_time_limit, remaining_time)))

        # Vizinhança Hamming(z, z_bar) <= k
        cols, coeffs = [], []
        ones_count = 0
        for idx, val in zip(bin_idxs, incumbent_sol):
            if val >= 0.5:
                cols.append(idx)
                coeffs.append(-1.0)
                ones_count += 1
            else:
                cols.append(idx)
                coeffs.append(1.0)

        rhs_upper = float(k - ones_count)
        h.addRow(-highspy.kHighsInf, rhs_upper, len(cols), cols, coeffs)

        h.run()
        info = h.getInfo()
        status = h.getModelStatus()

        if status in [highspy.HighsModelStatus.kOptimal, highspy.HighsModelStatus.kTimeLimit]:
            sol = h.getSolution()
            if sol.value_valid and info.objective_function_value < (best_obj - 1e-4):
                best_obj = info.objective_function_value
                incumbent_sol = np.round([sol.col_value[idx] for idx in bin_idxs])
                k = k_min
                continue

        k += 4
        if k > k_max:
            break  # Estagnou na busca global -> Passa para o Polimento Fix-and-Optimize

    # 3. Fase 2: Polimento por Fix-and-Optimize em Janelas Temporais Móveis
    # Fixa a maior parte dos setups e resolve subproblemas minúsculos para refinar custos
    step_size = max(4, window_size // 2)
    for w_start in range(0, N, step_size):
        remaining_time = time_limit - (time.time() - start_time)
        if remaining_time <= 0.5:
            break

        w_end = min(w_start + window_size, N)
        h_fo, bin_idxs, _ = build_ulsr_model(datafile)
        h_fo.setOptionValue("output_flag", debug)
        h_fo.setOptionValue("presolve", "on")
        h_fo.setOptionValue("time_limit", float(min(2.0, remaining_time)))  # 2s por janela

        # Fixa os setups fora da janela temporal atual na solução incumbente
        idx_y = bin_idxs[:N]
        idx_yr = bin_idxs[N:]

        for t in range(N):
            if not (w_start <= t < w_end):
                val_y = incumbent_sol[t]
                val_yr = incumbent_sol[N + t]
                h_fo.changeColBounds(idx_y[t], float(val_y), float(val_y))
                h_fo.changeColBounds(idx_yr[t], float(val_yr), float(val_yr))

        h_fo.run()
        info_fo = h_fo.getInfo()
        status_fo = h_fo.getModelStatus()

        if status_fo in [highspy.HighsModelStatus.kOptimal, highspy.HighsModelStatus.kTimeLimit]:
            sol_fo = h_fo.getSolution()
            if sol_fo.value_valid and info_fo.objective_function_value < (best_obj - 1e-4):
                best_obj = info_fo.objective_function_value
                incumbent_sol = np.round([sol_fo.col_value[idx] for idx in bin_idxs])

    exec_time = time.time() - start_time
    gap = (best_obj - best_bound) / abs(best_obj) if (best_bound is not None and best_obj is not None and best_obj != 0) else None

    return {
        'INSTANCIA': os.path.basename(datafile),
        'BOUND': best_bound,
        'OBJVAL': best_obj,
        'GAP': gap,
        'TIME': exec_time,
        'OTIMALIDADE': 'LB-EDD+ (Hybrid)'
    }

In [18]:
# CÉLULA 3: Execução do Experimento 2 Avançado (LB-EDD+)
import os
import re
import glob
import pandas as pd
from IPython.display import display

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass

def chave_natural(caminho):
    nome = os.path.basename(caminho)
    partes = re.split(r'(\d+)', nome)
    return [int(p) if p.isdigit() else p for p in partes]

pasta          = "/content/drive/MyDrive/Colab Notebooks/sifaleras"
extensao       = "*.txt"
time_limit     = 300.0

inicio = 1
fim    = 2  # Ajuste para a sua bateria de testes

caminho_busca = os.path.join(pasta, extensao)
todos_arquivos = sorted(glob.glob(caminho_busca), key=chave_natural)
arquivos = todos_arquivos[inicio-1 : fim]

print(f"--- EXPERIMENTO 2 (LB-EDD+): Local Branching + Fix-and-Optimize ---")
print(f"Instâncias a processar: {len(arquivos)} | Tempo Limite Max={time_limit}s\n")

if len(arquivos) == 0:
    print("❌ Nenhum arquivo .txt foi encontrado!")
else:
    resultados = []
    for arq in arquivos:
        print(f"A processar: {os.path.basename(arq)}...", end=" ", flush=True)
        res = ulsr_advanced_local_branching(
            arq,
            time_limit=time_limit,
            init_time=15.0,
            sub_time_limit=4.0,
            k_min=6,
            k_max=14,
            window_size=12,
            debug=False
        )
        print("Concluído!")
        resultados.append(res)

    df = pd.DataFrame(resultados)

    tabela = pd.DataFrame({
        'INSTANCIA':   df['INSTANCIA'],
        'BOUND':       df['BOUND'].map(lambda v: f"{v:.2f}" if pd.notnull(v) and isinstance(v, (int, float)) else "N/A"),
        'OBJVAL':      df['OBJVAL'].map(lambda v: f"{v:.2f}" if pd.notnull(v) and isinstance(v, (int, float)) else "N/A"),
        'GAP':         df['GAP'].map(lambda v: f"{v*100:.3f}%" if pd.notnull(v) and isinstance(v, (int, float)) else "N/A"),
        'TIME':        df['TIME'].map(lambda v: f"{v:.4f}" if pd.notnull(v) and isinstance(v, (int, float)) else "N/A"),
        'OTIMALIDADE': df['OTIMALIDADE'],
    })

    display(tabela)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- EXPERIMENTO 2 (LB-EDD+): Local Branching + Fix-and-Optimize ---
Instâncias a processar: 2 | Tempo Limite Max=300.0s

A processar: 52_1.txt... Concluído!
A processar: 52_2.txt... Concluído!


,INSTANCIA,BOUND,OBJVAL,GAP,TIME,OTIMALIDADE
0,52_1.txt,8586.16,8698.80,1.295%,29.9058,LB-EDD+ (Hybrid)
1,52_2.txt,8588.17,8806.40,2.478%,43.3992,LB-EDD+ (Hybrid)
